In [1]:
!pip install langchain langchain-community chromadb sentence-transformers groq pypdf streamlit -q

In [2]:
!pip install groq --upgrade -q

In [3]:
import langchain
import chromadb
import groq
print("All libraries are imported successfully")

All libraries are imported successfully


In [4]:
import os
os.environ["GROQ_API_KEY"] = "your-api-key-here"

In [5]:
from google.colab import files
uploaded=files.upload()

Saving RAG_assistant.pdf to RAG_assistant.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Loading pdf
loader=PyPDFLoader("RAG_assistant.pdf")
pages=loader.load()
print(f"Total pages:{len(pages)}")

#spliting to chunks
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=splitter.split_documents(pages)
print(f"Total chunks:{len(chunks)}")

/tmp/ipykernel_39881/225190876.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total pages:1
Total chunks:9


In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

#creating embed model
embeddings=HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

#storing chunks as embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print("embeddings are created and stored correctly")

/tmp/ipykernel_39881/3740728270.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddings are created and stored correctly


In [8]:
from langchain_groq import ChatGroq

# Connect to Groq LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

# Create retriever
retriever = vectorstore.as_retriever()

# Function to answer questions
def ask_question(question):
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""Use the following context to answer the question.

Context: {context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    return response.content

print("QA system ready!")

QA system ready!


In [9]:
print(ask_question("What is Medical Image Analysis"))

Medical Image Analysis is the process of using AI-powered image analysis systems to examine medical images such as X-rays, MRI scans, CT scans, and pathology slides to detect diseases with high accuracy. These systems are trained on millions of medical images to identify patterns that indicate cancer, fractures, infections, and other conditions.


In [10]:
# Interactive QA loop
while True:
    question = input("Ask a question (or type 'quit' to stop): ")
    if question.lower() == 'quit':
        break
    answer = ask_question(question)
    print(f"Answer: {answer}\n")

Ask a question (or type 'quit' to stop): wt is drug disease
Answer: Based on the context provided, it appears that "heart disease" is an example of a drug disease. The text mentions that AI models can be used to analyze patient data, including medical history, to predict and prevent diseases such as diabetes, heart disease, and kidney failure.

Ask a question (or type 'quit' to stop): drug discovery
Answer: Drug Discovery

Ask a question (or type 'quit' to stop): what is disease prediction
Answer: Disease prediction is the process of using machine learning models to analyze patient data, including age, medical history, lifestyle factors, and lab results, to predict the likelihood of a patient developing certain diseases such as diabetes, heart disease, and kidney failure.

Ask a question (or type 'quit' to stop): what is traditional drug discovery
Answer: Traditional drug discovery takes 10 to 15 years and costs billions of dollars.

Ask a question (or type 'quit' to stop): wt is medic